# ViNumQA 1-shot -- GLM-5.2 via FPT AI Factory

Same structure as `few-shot/vsf-few-shot-vinumqa-glm5.2.ipynb`, with the
few-shot prefix replaced by the single exemplar used across the team's 1-shot
notebooks, so 1-shot and few-shot numbers stay comparable for this model.

The endpoint-specific plumbing is carried over from the few-shot notebook
because it was established there against real failures, not guessed:

1. **Rate limits** (RPM=50, TPM=100,000). Measured over 15 real probe requests
   in `few-shot/vsf-few-shot-vinumqa-glm5.2-rate-check.ipynb`: mean
   prompt_tokens=2407, mean total_tokens=2721/request, so **TPM binds before
   RPM**, capping the loop at ~36.7 req/min. A 1-shot prompt is shorter than a
   3-shot one, so it will sit somewhat under that ceiling -- the sliding-window
   `RateLimiter` adapts on its own, since it throttles on the real
   `usage.total_tokens` returned per response rather than a fixed sleep.

   This replaces the `ThreadPoolExecutor(max_workers=8)` the 1-shot notebook
   previously used. Eight unthrottled concurrent requests cannot respect
   RPM=50/TPM=100k, and the few-shot run died at 73/497 on a `RateLimitError`
   before the limiter was added.

2. **Reasoning output**. GLM-5.2 returns its chain-of-thought in a separate
   `message.reasoning_content` field, so `message.content` is already the bare
   program (verified in the few-shot notebook: `content` came back as a clean
   `'add(2408, 1364)'`). No stripping is needed, and the reasoning-model probe
   cell that used to live here is gone -- it spent an API call rediscovering
   this. `usage.completion_tokens` already counts the reasoning tokens, so the
   limiter accounts for them correctly.

3. **Checkpoint after every request**, not every N. Losing a 429-interrupted
   run part-way costs nothing already generated.


In [1]:
import pandas as pd
from pathlib import Path
from tabulate import tabulate

# Robust to whichever working directory Jupyter starts in (repo root vs.
# the notebook's own folder, which is VSCode's default for .ipynb files).
_CANDIDATES = [
    Path("datasets/ViNumQA/test.json"),
    Path("../../../datasets/ViNumQA/test.json"),
]
DATA_PATH = next((p for p in _CANDIDATES if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        f"Could not find test.json. cwd={Path.cwd()}, tried: {[str(p) for p in _CANDIDATES]}"
    )
print(f"Loading data from: {DATA_PATH.resolve()}")

df = pd.read_json(DATA_PATH)
df.sample(n=5)

Loading data from: D:\NumReasoning4VietnameseFinancialText\datasets\ViNumQA\test.json


,pre_text,table,post_text,id,qa
360,[thảo luận và phân tích của ban quản lý 150 jp...,"[[ngày 31 tháng 12, (triệu đô la), ngày 31 thá...",[các khoản phải thu từ khách hàng và các khoản...,JPM/2012/page_140.pdf-1,{'question': 'Tỷ lệ thay đổi phần trăm trong c...
89,"[định giá và khuyến nghị, chúng tôi duy trì kh...","[[FY (tỷ đồng), FY18, FY19, FY20, FY21, FY22],...",[.],masvn/2020/2020163-VN_DHG_Update_BUY_MAS_20201...,{'question': 'Tính mức giảm của ROE từ năm 202...
200,"[sử dụng phương pháp p/b và rnav để định giá, ...","[[Năm tài chính (31/12), 2018, 2019, 2020 (Dự ...",[.],masvn/2020/2020193-VN_IndustrialRealEstate_Upd...,{'question': 'Tính trung bình tỷ suất lợi nhuậ...
481,[.],"[[, 1Q18, 1Q19, YoY (%), Kế hoạch 2019 (%)], [...",[doanh thu tăng trưởng trong quý đầu năm có ph...,masvn/2020/2020013-VN_18042019MWGShortNote_TV_...,{'question': 'Tốc độ tăng trưởng doanh thu quý...
361,"[đánh giá kết quả 2018, năm 2018, mbb báo cáo ...","[[Năm tài chính, 2013, 2014, 2015, 2016, 2017,...",[.],masvn/2020/2020010-190314_MBB_2018review_VN/pa...,"{'question': 'năm 2019, tỷ lệ P/B mục tiêu là ..."


In [2]:
len(df)

497

In [3]:
def formatting_pre_text(sample):
    return "\n".join(sample["pre_text"])

def formatting_table(sample):
    return tabulate(sample["table"][1:], headers=sample["table"][0], tablefmt="github")

def formatting_post_text(sample):
    return "\n".join(sample["post_text"])

def processing_input_question(sample):
    return sample["qa"]["question"]

def processing_program_content(sample):
    return sample["qa"]["program"]

def processing_answer_content(sample):
    return sample["qa"]["exe_ans"]

df["pre_text_processed"] = df.apply(lambda x: formatting_pre_text(x), axis=1)
df["post_text_processed"] = df.apply(lambda x: formatting_post_text(x), axis=1)
df["table_processed"] = df.apply(lambda x: formatting_table(x), axis=1)
df["table_raw"] = df["table"]  # keep raw rows for table_* row-name lookup at eval time
df["input_question"] = df.apply(lambda x: processing_input_question(x), axis=1)
df["program_processed"] = df.apply(lambda x: processing_program_content(x), axis=1)
df["answer_processed"] = df.apply(lambda x: processing_answer_content(x), axis=1)
df.sample(n=5)

,pre_text,table,post_text,id,qa,pre_text_processed,post_text_processed,table_processed,table_raw,input_question,program_processed,answer_processed
236,"[định giá và khuyến nghị, chúng tôi khuyến ngh...","[[Năm tài chính, 2017, 2018, 2019, 2020F, 2021...",[.],masvn/2020/2020169-VHMQ32020/page_1_QA4,{'question': 'Giá trị EPS (VND) cao nhất là ba...,định giá và khuyến nghị\nchúng tôi khuyến nghị...,.,| Năm tài chính | 2017 | 2018 ...,"[[Năm tài chính, 2017, 2018, 2019, 2020F, 2021...",Giá trị EPS (VND) cao nhất là bao nhiêu?,"table_max(EPS (VND), none)",11133.0
170,"[doanh thu năm 2019 đạt 3,298 tỷ (+14 % yoy) v...","[[(Tỷ đồng), FY 2015, FY 2016, FY 2017, FY 201...",[.],masvn/2020/2020060-TLG_Companynote_MAS03.06.20...,{'question': 'Tính tỷ lệ tăng trưởng doanh thu...,"doanh thu năm 2019 đạt 3,298 tỷ (+14 % yoy) và...",.,| (Tỷ đồng) | FY 2015 | FY 2016 |...,"[[(Tỷ đồng), FY 2015, FY 2016, FY 2017, FY 201...",Tính tỷ lệ tăng trưởng doanh thu từ năm 2018 đ...,"subtract(3252, 2856), divide(#0, 2856)",0.13866
289,"[sự hấp dẫn đến từ báo cáo tài chính, mặc dù s...","[[, Lợi nhuận, Doanh thu], [VN-Index, 23.13, 2...",[sự khác biệt giữa mức tăng trưởng doanh thu v...,masvn/2020/2020007-RealEstatestocks_VaR_V_fina...,{'question': 'Tăng trưởng doanh thu trung bình...,sự hấp dẫn đến từ báo cáo tài chính\nmặc dù số...,sự khác biệt giữa mức tăng trưởng doanh thu và...,| ...,"[[, Lợi nhuận, Doanh thu], [VN-Index, 23.13, 2...",Tăng trưởng doanh thu trung bình của ngành Y t...,"add(6.05, 6.46), divide(#0, 2)",6.255
349,"[định giá và khuyến nghị, chúng tôi khuyến ngh...","[[Năm tài chính, 2017, 2018, 2019, 2020F, 2021...",[.],masvn/2020/2020169-VHMQ32020/page_1_QA2,{'question': 'Tổng lợi nhuận hoạt động dự kiến...,định giá và khuyến nghị\nchúng tôi khuyến nghị...,.,| Năm tài chính | 2017 | 2018 ...,"[[Năm tài chính, 2017, 2018, 2019, 2020F, 2021...",Tổng lợi nhuận hoạt động dự kiến từ năm 2020 đ...,"add(36609, 41879), add(#0, 47268)",125756.0
107,[rủi ro tín dụng tập trung các công cụ tài chí...,"[[, 2010, 2009, 2008], [số dư đầu năm, $ 55.2,...","[vào năm 2008, sau thương vụ mua lại allied, c...",RSG/2010/page_98.pdf-2,{'question': 'Tính đến ngày 31 tháng 12 năm 20...,rủi ro tín dụng tập trung các công cụ tài chín...,"vào năm 2008, sau thương vụ mua lại allied, ch...",| | 2010 ...,"[[, 2010, 2009, 2008], [số dư đầu năm, $ 55.2,...","Tính đến ngày 31 tháng 12 năm 2010, tỷ lệ tiền...","divide(172.8, 50.9)",3.39489


In [4]:
df = df[["pre_text_processed", "table_processed", "table_raw", "post_text_processed", "input_question", "program_processed", "answer_processed"]]
# NOTE: use a flat list here, not [[...]] — the double brackets create a
# MultiIndex column (each name becomes a 1-tuple like ('question',)), which
# silently breaks boolean filtering like df["generated_program"] == "" later
# (df["col"] returns a 1-column DataFrame instead of a Series in that case).
df.columns = ["pre_text", "table", "table_raw", "post_text", "question", "program", "answer"]
df["generated_program"] = ""
df["calculated_program"] = ""
df

,pre_text,table,table_raw,post_text,question,program,answer,generated_program,calculated_program
0,thuyết minh báo cáo tài chính hợp nhất ( tiếp ...,| các thành phần của ảnh hưởng lũy kế của việc...,[[các thành phần của ảnh hưởng lũy kế của việc...,.,Sự thay đổi trong thu nhập ròng từ hiệu ứng tí...,"add(30, 1)",31.0,,
1,định giá và khuyến nghị:\nchúng tôi khuyến ngh...,| Năm tài chính (31/12) | FY17 | FY1...,"[[Năm tài chính (31/12), FY17, FY18, FY19, FY2...",.,"Theo dự phóng, doanh thu và lợi nhuận ròng quý...","subtract(9829, 642)",9187.0,,
2,"sử dụng phương pháp p/b và rnav để định giá, c...",| Năm tài chính (31/12) | 2016 | 2017 | ...,"[[Năm tài chính (31/12), 2016, 2017, 2018, 201...",.,IDC có bao nhiêu ha quỹ đất sẵn sàng cho thuê ...,"add(495, 398)",893.0,,
3,27/10/13 26/10/14 25/10/15 30/10/16 29/10/17 2...,| | 27/10/2013 |...,"[[, 27/10/2013, 26/10/2014, 25/10/2015, 30/10/...",.,Tỷ suất lợi nhuận trên đầu tư (ROI) của Applie...,"subtract(96.67, 100), divide(#0, 100)",-0.0333,,
4,thông tin tài chính bổ sung hiệu suất cổ phiếu...,| | 12/26/08 | ...,"[[, 12/26/08, 12/31/09, 12/31/10, 12/31/11, 12...",218 báo cáo thường niên năm 2013 của goldman s...,tỷ lệ lợi nhuận tích lũy tổng cộng theo phần t...,"subtract(248.36, 100), divide(#0, 100)",1.4836,,
...,...,...,...,...,...,...,...,...,...
492,thuyết minh báo cáo tài chính hợp nhất năm 201...,| ...,"[[, 2008, 2007], [Số dư đầu kỳ, $ 134.8, $ 266...",trong tổng số lợi ích thuế chưa được ghi nhận ...,Tỷ lệ phần trăm lợi ích thuế chưa được công nh...,"divide(131.8, 148.8)",0.88575,,
493,định giá và khuyến nghị:\nchúng tôi khuyến ngh...,| Năm tài chính (31/12) | FY17 | FY1...,"[[Năm tài chính (31/12), FY17, FY18, FY19, FY2...",.,Doanh thu trung bình từ năm tài chính 2017 đến...,"add(29710, 32662), add(#0, 35374), divide(#1, 3)",32582.0,,
494,định giá và khuyến nghị:\nchúng tôi khuyến ngh...,| Năm tài chính (31/12) | FY17 | FY1...,"[[Năm tài chính (31/12), FY17, FY18, FY19, FY2...",.,Tính phần trăm thay đổi EPS từ năm 2020 đến 2021.,"subtract(962, 788), divide(#0, 788)",0.22081,,
495,"trong quá trình kinh doanh thông thường, dựa t...",| ( đơn vị: nghìn ) | diện tích ròng chưa ph...,"[[( đơn vị: nghìn ), diện tích ròng chưa phát ...",( a ) một giếng khoan thăm dò được lên kế hoạc...,Tỷ lệ phần trăm diện tích đất chưa phát triển ...,"divide(145, 586)",0.24744,,


In [5]:
SYSTEM_MESSAGE = """You are a financial analysis AI. Your task is to generate a sequential computation program to answer the question, based on the provided context.

### LIST OF 10 VALID OPERATORS:

1. add(a, b) -> a + b
2. subtract(a, b) -> a - b
3. multiply(a, b) -> a * b
4. divide(a, b) -> a / b
5. exp(a, b) -> a^b
6. greater(a, b) -> 1.0 if a > b, else 0.0
7. table_sum(row_name, none) -> sum of the numeric values in the table row named `row_name`
8. table_average(row_name, none) -> arithmetic mean of the numeric values in the table row named `row_name`
9. table_max(row_name, none) -> maximum of the numeric values in the table row named `row_name`
10. table_min(row_name, none) -> minimum of the numeric values in the table row named `row_name`

### RULES:
- Do not use free-form mathematical symbols ("+", "-", "*", "/") outside of parentheses. Every calculation must use one of the 10 operators above.
- table_* operators take exactly two arguments: the row name (copied exactly as it appears as the first cell of the target row) and the literal `none` (e.g. table_max(Lãi ròng, none)), never a list of numeric values.
- Do not perform mental calculations or provide explanations. The output must contain only the program string.
- Reference the result of a previous step using #0 (step 1), #1 (step 2), etc. Steps are separated by commas.
- Preserve the original number format from the context. If a value is missing, use 'none'."""

USER_MESSAGE_FRAME = """### CONTEXT:
[TEXT BEFORE TABLE]
{pre_text}

[TABLE]
{table}

[TEXT AFTER TABLE]
{post_text}

### QUESTION:
{question}

### PROGRAM:"""

# One-shot exemplar (from Figure 1 of the VLSP 2025 NumQA paper): a single
# worked Table-Only example prepended as a user/assistant turn before the
# real query. Kept identical to the exemplar used in the team's Qwen3-4B
# 1-shot notebook (notebooks/vinumqa/1-shot/1shot_qwen4B.ipynb on the `chi`
# branch) so results are comparable across models/APIs at the same shot count.
ONE_SHOT_EXAMPLE_TABLE = tabulate(
    [["Net revenue", "391", "914"], ["Gross profit", "163", "513"]],
    headers=["Metric (VND billion)", "2022", "2023"],
    tablefmt="github",
)

ONE_SHOT_EXAMPLE_USER = USER_MESSAGE_FRAME.format(
    pre_text=(
        "hệ số nợ trên vốn chủ sở hữu (d/e) của công ty cổ phần phát triển đô thị "
        "từ liêm là 0.1 lần và thường xuyên ở mức thấp, tránh rủi ro thanh khoản "
        "cho doanh nghiệp."
    ),
    table=ONE_SHOT_EXAMPLE_TABLE,
    post_text=".",
    question="Doanh thu thuần năm 2023 gấp bao nhiêu lần doanh thu thuần năm 2022?",
)
ONE_SHOT_EXAMPLE_ASSISTANT = "divide(914, 391)"

In [6]:
import os
import time
from collections import deque
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / ".git").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

API_KEY = os.environ["API_KEY"]
BASE_URL = os.environ["BASE_URL"]
MODEL = "GLM-5.2"

# max_retries=0: disable the OpenAI SDK's own built-in retry-on-429 (it retries
# with its own back-off, invisible to RateLimiter below -- those retries hit the
# server without ever calling limiter.record(), so the limiter's window
# under-counts what was really sent and the cap gets blown from underneath it).
# All retry logic is handled explicitly in the generation loop instead, so every
# request -- first try or retry -- goes through the limiter.
client = OpenAI(api_key=API_KEY, base_url=BASE_URL, max_retries=0)

RPM_LIMIT = 50
TPM_LIMIT = 100_000


class RateLimiter:
    """Sliding 60s window limiter for RPM and TPM.

    Call `wait_if_needed(estimated_tokens)` before each request (blocks via
    time.sleep if the window is already at capacity), then `record(actual_tokens)`
    once the response arrives with the real usage.total_tokens.
    """

    def __init__(self, rpm_limit: int, tpm_limit: int, window_s: float = 60.0):
        self.rpm_limit = rpm_limit
        self.tpm_limit = tpm_limit
        self.window_s = window_s
        self.request_times = deque()
        self.token_events = deque()

    def _prune(self, now: float):
        while self.request_times and now - self.request_times[0] > self.window_s:
            self.request_times.popleft()
        while self.token_events and now - self.token_events[0][0] > self.window_s:
            self.token_events.popleft()

    def wait_if_needed(self, estimated_tokens: int):
        while True:
            now = time.monotonic()
            self._prune(now)

            tokens_in_window = sum(t for _, t in self.token_events)
            requests_in_window = len(self.request_times)

            over_rpm = requests_in_window >= self.rpm_limit
            over_tpm = tokens_in_window + estimated_tokens > self.tpm_limit

            if not over_rpm and not over_tpm:
                return

            oldest = min(
                self.request_times[0] if self.request_times else float("inf"),
                self.token_events[0][0] if self.token_events else float("inf"),
            )
            sleep_for = max(0.05, self.window_s - (now - oldest))
            time.sleep(sleep_for)

    def record(self, actual_tokens: int):
        now = time.monotonic()
        self.request_times.append(now)
        self.token_events.append((now, actual_tokens))

    def on_rate_limit_error(self, retry_after=None):
        """Force the window to look full for retry_after seconds (or the whole
        window if the server gave no Retry-After hint). A 429 is itself proof
        that the local accounting has drifted from the server's real state, so
        the safe correction is to treat the window as saturated rather than
        trust the (evidently wrong) count.
        """
        now = time.monotonic()
        self.request_times.clear()
        self.token_events.clear()
        pause = retry_after if retry_after is not None else self.window_s
        self.request_times.append(now + pause - self.window_s)
        self.token_events.append((now + pause - self.window_s, self.tpm_limit))


limiter = RateLimiter(rpm_limit=RPM_LIMIT, tpm_limit=TPM_LIMIT)
print(f"Model: {MODEL} | RPM<={RPM_LIMIT} | TPM<={TPM_LIMIT:,}")


Model: GLM-5.2 | RPM<=50 | TPM<=100,000


In [22]:
import json
import re
from tqdm import tqdm
from openai import RateLimitError

# Anchored to the repo root so the checkpoint lands in the same place whether
# Jupyter starts in the repo root or in this notebook's own folder.
CHECKPOINT_PATH = PROJECT_ROOT / "notebooks" / "vinumqa" / "1-shot" / "vsf-1-shot-GLM_5.2.checkpoint.json"
MAX_RETRIES_PER_SAMPLE = 5
DEFAULT_RETRY_AFTER_S = 65.0  # the 429 seen on this endpoint said "try again in 60s" -- pad slightly

MAX_TOKENS = 8192  # GLM-5.2 is a reasoning model: content is short, but the
                   # hidden reasoning it bills for is not.

# The 1-shot prefix: one user/assistant pair, using the same USER_MESSAGE_FRAME
# as the real query so the exemplar and the query are formatted identically.
one_shot_messages = [
    {"role": "user", "content": ONE_SHOT_EXAMPLE_USER},
    {"role": "assistant", "content": ONE_SHOT_EXAMPLE_ASSISTANT},
]

# Resume support: previously-generated outputs are keyed by the dataframe's
# integer index (stored as a string in JSON) and skipped below.
if CHECKPOINT_PATH.exists():
    with open(CHECKPOINT_PATH, "r", encoding="utf-8") as f:
        checkpoint = {int(k): v for k, v in json.load(f).items()}
    print(f"Resuming from checkpoint: {len(checkpoint)} samples already generated.")
else:
    checkpoint = {}

for df_index, output in checkpoint.items():
    df.at[df_index, "generated_program"] = output


def _extract_retry_after(err):
    header_val = err.response.headers.get("retry-after") if err.response is not None else None
    if header_val is not None:
        try:
            return float(header_val)
        except ValueError:
            pass
    # Fallback: the FPT AI Factory error body embeds it in the message text,
    # e.g. "...Please try again in 60s."
    match = re.search(r"try again in (\d+(?:\.\d+)?)s", str(err))
    return float(match.group(1)) if match else None


pending_indices = [idx for idx in df.index if idx not in checkpoint]
n_truncated = 0

for df_index in tqdm(pending_indices, desc="Generating program..."):
    values = df.loc[df_index]

    messages = [
        {"role": "system", "content": SYSTEM_MESSAGE},
        *one_shot_messages,
        {"role": "user", "content": USER_MESSAGE_FRAME.format(
            pre_text=values["pre_text"], table=values["table"],
            post_text=values["post_text"], question=values["question"],
        )},
    ]

    # Estimate tokens before the call (the real usage.prompt_tokens isn't known
    # until the response arrives) -- a rough chars/4 heuristic is enough, since
    # limiter.record() corrects the window with the real total straight after,
    # so any under/over-estimate only affects this one wait.
    est_tokens = sum(len(m["content"]) for m in messages) // 4 + MAX_TOKENS

    output = None
    for attempt in range(1, MAX_RETRIES_PER_SAMPLE + 1):
        limiter.wait_if_needed(est_tokens)
        try:
            chat_completion = client.chat.completions.create(
                model=MODEL,
                messages=messages,
                temperature=0.0,
                max_tokens=MAX_TOKENS,
                stream=False,
            )
        except RateLimitError as err:
            retry_after = _extract_retry_after(err) or DEFAULT_RETRY_AFTER_S
            print(f"\n[df_index={df_index}] RateLimitError on attempt {attempt}/{MAX_RETRIES_PER_SAMPLE}: "
                  f"{err}. Sleeping {retry_after:.0f}s and resetting the limiter window before retrying.")
            limiter.on_rate_limit_error(retry_after)
            time.sleep(retry_after)
            continue

        limiter.record(chat_completion.usage.total_tokens)

        choice = chat_completion.choices[0]
        # message.content is the final program string; GLM-5.2's chain-of-thought
        # is returned separately in message.reasoning_content, not mixed in here.
        output = (choice.message.content or "").strip().strip("\n")
        if choice.finish_reason == "length":
            # Not fatal and not retried: a truncated program simply fails to
            # tokenize and scores 0 later. Counted so a systematically-too-small
            # MAX_TOKENS shows up as a number rather than as quiet zeros.
            n_truncated += 1
        break

    if output is None:
        print(f"\n[df_index={df_index}] Giving up after {MAX_RETRIES_PER_SAMPLE} attempts -- leaving generated_program empty.")
        output = ""

    df.at[df_index, "generated_program"] = output
    checkpoint[df_index] = output
    with open(CHECKPOINT_PATH, "w", encoding="utf-8") as f:
        json.dump({str(k): v for k, v in checkpoint.items()}, f, ensure_ascii=False, indent=2)

print(f"Done. {(df['generated_program'] != '').sum()} / {len(df)} samples generated.")
if n_truncated:
    print(f"WARNING: {n_truncated} responses hit finish_reason='length' at MAX_TOKENS={MAX_TOKENS} "
          f"-- those programs are probably cut off and will score 0.")


Resuming from checkpoint: 303 samples already generated.


Generating program...:   0%|          | 0/194 [00:00<?, ?it/s]

Generating program...: 100%|██████████| 194/194 [22:43<00:00,  7.03s/it]


Done. 489 / 497 samples generated.


In [23]:
!pip install -q sympy
import sys
sys.path.insert(0, "../../evaluate")  # scorer.py lives at notebooks/evaluate/scorer.py

from scorer import evaluate_dataframe  # noqa: E402

# scorer.py is the shared ViNumQA evaluator (notebooks/evaluate/scorer.py): it
# ports FinQA's official evaluation protocol (sympy-based symbolic Program
# Accuracy, table-row-lookup-aware Execution Accuracy) instead of a
# hand-rolled parser, and correctly executes table_*(row_name, none) calls by
# looking up the named row in the raw table.


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [24]:
df_scored, summary = evaluate_dataframe(
    df,
    generated_col="generated_program",   # cột bạn đang ghi output model vào
    gold_program_col="program",
    gold_answer_col="answer",
    table_col="table_raw",
)

print(summary)  # {'program_accuracy': ..., 'execution_accuracy': ...}

{'program_accuracy': 0.48490945674044267, 'execution_accuracy': 0.5171026156941649}


In [25]:
import json

results_path = Path(f"outputs/1shot_{MODEL}_results.csv")
summary_path = Path(f"outputs/1shot_{MODEL}_summary.json")

df_scored.to_csv(results_path, index=False)
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump({"model": MODEL, "shot": "1-shot", "max_tokens": MAX_TOKENS, **summary}, f, ensure_ascii=False, indent=2)

print(f"Saved per-sample results to {results_path}")
print(f"Saved summary to {summary_path}")

Saved per-sample results to outputs\1shot_GLM-5.2_results.csv
Saved summary to outputs\1shot_GLM-5.2_summary.json
